# 02 - CSV para Delta Lake (MinIO)

Lê os CSVs do bucket **landing-zone** e grava cada um como tabela **Delta Lake** no bucket **bronze**.

**Pré-requisito:** notebook `01` executado.

> Para repetir o experimento do zero, basta executar este notebook de novo: o modo `overwrite`
> recria o conteúdo das tabelas Delta a partir dos CSVs.

## 1. Imports e configuração

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
if not (RAIZ / '.env').exists():
    raise FileNotFoundError('Arquivo .env não encontrado. Rode: cp .env.example .env')
load_dotenv(RAIZ / '.env', override=True)

True

## 2. SparkSession com Delta Lake e MinIO

In [2]:
from pyspark.sql import SparkSession
from delta import *
from delta.tables import DeltaTable

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

spark = (
    SparkSession.builder
    .appName('CSV_to_Delta')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.1,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    # MinIO / S3A
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada! Spark', spark.version)

26/10/12 18:38:49 WARN Utils: Your hostname, DESKTOP-73OKR5D resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/12 18:38:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/thomasmaggi/pesquisadados_parte2/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thomasmaggi/.ivy2/cache
The jars for the packages stored in: /home/thomasmaggi/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b8ee2abc-4c1c-46f3-8052-7f3234b28b82;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (667ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#

SparkSession criada! Spark 3.5.3


## 3. Criar o bucket bronze no MinIO

In [3]:
import boto3
from botocore.client import Config

s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] já existe')
except Exception:
    s3_client.create_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] criado!')

print('Buckets:', [b['Name'] for b in s3_client.list_buckets()['Buckets']])

Bucket [bronze] criado!
Buckets: ['bronze', 'landing-zone']


## 4. Listar os CSVs da landing-zone

In [4]:
resposta = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
csv_files = [o['Key'] for o in resposta.get('Contents', []) if o['Key'].endswith('.csv')]

print(f'{len(csv_files)} arquivos CSV encontrados no bucket [{LANDING_BUCKET}]:')
for f in csv_files:
    print(f'  - {f}')

2 arquivos CSV encontrados no bucket [landing-zone]:
  - clientes.csv
  - pedidos.csv


## 5. Ler os CSVs e gravar como Delta Lake

O schema é definido explicitamente (tipos corretos de `DATE` e `DECIMAL`), igual ao DDL do SQL Server.

In [5]:
from pyspark.sql.types import (StructType, StructField, IntegerType,
                               StringType, DecimalType, DateType)

schemas = {
    'clientes': StructType([
        StructField('id_cliente', IntegerType(), True),
        StructField('nome', StringType(), True),
        StructField('cidade', StringType(), True),
        StructField('uf', StringType(), True),
    ]),
    'pedidos': StructType([
        StructField('id_pedido', IntegerType(), True),
        StructField('id_cliente', IntegerType(), True),
        StructField('produto', StringType(), True),
        StructField('quantidade', IntegerType(), True),
        StructField('valor_unitario', DecimalType(10, 2), True),
        StructField('data_pedido', DateType(), True),
        StructField('status', StringType(), True),
    ]),
}

print(f'Convertendo {len(csv_files)} CSVs para Delta Lake...\n')

for csv_file in csv_files:
    tabela = csv_file.replace('.csv', '')
    csv_path = f's3a://{LANDING_BUCKET}/{csv_file}'
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'

    leitor = spark.read.option('header', 'true')
    if tabela in schemas:
        df = leitor.schema(schemas[tabela]).csv(csv_path)
    else:
        df = leitor.option('inferSchema', 'true').csv(csv_path)

    df.write.format('delta').mode('overwrite').save(delta_path)
    print(f'  {tabela}: {df.count()} registros | {len(df.columns)} colunas -> {delta_path}')

print(f'\nConversão concluída! {len(csv_files)} tabelas Delta no bucket [{BRONZE_BUCKET}].')

Convertendo 2 CSVs para Delta Lake...



26/10/12 18:40:57 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

  clientes: 4 registros | 4 colunas -> s3a://bronze/clientes
  pedidos: 5 registros | 7 colunas -> s3a://bronze/pedidos

Conversão concluída! 2 tabelas Delta no bucket [bronze].


## 6. Validação — ler as tabelas Delta

In [6]:
for csv_file in csv_files:
    tabela = csv_file.replace('.csv', '')
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'

    eh_delta = DeltaTable.isDeltaTable(spark, delta_path)
    df_delta = spark.read.format('delta').load(delta_path)
    print(f'  {tabela}: Delta={eh_delta} | {df_delta.count()} registros')
    df_delta.printSchema()
    df_delta.show()

26/10/12 18:41:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

  clientes: Delta=True | 4 registros
root
 |-- id_cliente: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- uf: string (nullable = true)



+----------+----------+-------------+---+
|id_cliente|      nome|       cidade| uf|
+----------+----------+-------------+---+
|         1| Ana Souza|     Criciúma| SC|
|         2|Bruno Lima|Florianópolis| SC|
|         3|Carla Dias| Porto Alegre| RS|
|         4|Diego Melo|     Curitiba| PR|
+----------+----------+-------------+---+



  pedidos: Delta=True | 5 registros
root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- produto: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- valor_unitario: decimal(10,2) (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- status: string (nullable = true)



[Stage 35:===================================>                    (32 + 8) / 50]

+---------+----------+--------+----------+--------------+-----------+---------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|   status|
+---------+----------+--------+----------+--------------+-----------+---------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|     PAGO|
|      102|         2|   Mouse|         2|         80.00| 2026-09-02|     PAGO|
|      103|         3| Monitor|         1|       1200.00| 2026-09-03| PENDENTE|
|      104|         1| Teclado|         1|        250.00| 2026-09-05| PENDENTE|
|      105|         4| Headset|         1|        300.00| 2026-09-06|CANCELADO|
+---------+----------+--------+----------+--------------+-----------+---------+



In [7]:
resposta = s3_client.list_objects_v2(Bucket=BRONZE_BUCKET, Prefix='pedidos/')
print(f'Arquivos da tabela pedidos no bucket [{BRONZE_BUCKET}]:')
for obj in resposta.get('Contents', []):
    print(f'  {obj["Key"]}')

Arquivos da tabela pedidos no bucket [bronze]:
  pedidos/_delta_log/00000000000000000000.json
  pedidos/_delta_log/_commits/
  pedidos/part-00000-ebd829b7-ab86-409b-8ae3-52026b5e6ed4-c000.snappy.parquet


In [8]:
spark.stop()
print('SparkSession finalizada.')

SparkSession finalizada.
